In [1]:
from copy import deepcopy
import json

# Clinical review မလုပ်ရသေးသော prototype
RULESET_METADATA = {
    "name": "HabitatPulse Headache Review",
    "version": "0.1-draft",
    "status": "draft_not_clinically_reviewed",
    "purpose": "Evidence organization and medical review support",
    "validated_for_spaceflight": False,
    "produces_disease_probabilities": False,
}

SOURCES = {
    "NICE_CG150_1_1_1": {
        "title": "Headache assessment",
        "section": "Recommendation 1.1.1",
        "url": (
            "https://www.nice.org.uk/guidance/"
            "cg150/chapter/Recommendations"
        ),
        "scope": "Terrestrial guideline; requires adaptation and review",
    },
    "NASA_CO2_REV_D": {
        "title": "OCHMO-TB-004 Carbon Dioxide",
        "revision": "D",
        "date": "2026-03-10",
        "url": (
            "https://www.nasa.gov/wp-content/"
            "uploads/2023/12/ochmo-tb-004-carbon-dioxide.pdf"
        ),
        "scope": "Reference brief; not an individual diagnostic rule",
    },
}

CHECKIN_TEMPLATE = {
    "crew_id": None,
    "recorded_at_utc": None,

    "headache": {
        "present": None,
        "onset_at_utc": None,
        "severity_0_to_10": None,
        "duration_minutes": None,

        # Initial review questions; not a complete screening checklist
        "sudden_peak_within_5_minutes": None,
        "new_neurological_change": None,
        "impaired_consciousness": None,
        "worsening_with_fever": None,

        "associated_symptoms": None,
    },

    "daily_context": {
        "sleep_hours": None,
        "fluid_intake_ml": None,
        "recent_activity": None,
        "location_id": None,
    },

    "history": {
        "previous_similar_headaches": None,
        "known_conditions": None,
        "current_medications": None,
        "allergies": None,
    },

    "environment": {
        "measured_at_utc": None,
        "location_id": None,
        "source_id": None,
        "is_simulated": None,
        "co2_ppm": None,
        "cabin_pressure_kpa": None,
        "exposure_duration_minutes": None,
        "anomaly_flag": None,
        "anomaly_model_version": None,
    },
}

checkin = deepcopy(CHECKIN_TEMPLATE)

print("Ruleset:", RULESET_METADATA["version"])
print("Status:", RULESET_METADATA["status"])
print(json.dumps(checkin, indent=2, ensure_ascii=False))

Ruleset: 0.1-draft
Status: draft_not_clinically_reviewed
{
  "crew_id": null,
  "recorded_at_utc": null,
  "headache": {
    "present": null,
    "onset_at_utc": null,
    "severity_0_to_10": null,
    "duration_minutes": null,
    "sudden_peak_within_5_minutes": null,
    "new_neurological_change": null,
    "impaired_consciousness": null,
    "worsening_with_fever": null,
    "associated_symptoms": null
  },
  "daily_context": {
    "sleep_hours": null,
    "fluid_intake_ml": null,
    "recent_activity": null,
    "location_id": null
  },
  "history": {
    "previous_similar_headaches": null,
    "known_conditions": null,
    "current_medications": null,
    "allergies": null
  },
  "environment": {
    "measured_at_utc": null,
    "location_id": null,
    "source_id": null,
    "is_simulated": null,
    "co2_ppm": null,
    "cabin_pressure_kpa": null,
    "exposure_duration_minutes": null,
    "anomaly_flag": null,
    "anomaly_model_version": null
  }
}


In [2]:
# Initial subset only — complete screening checklist မဟုတ်ပါ
HEADACHE_REVIEW_RULES = [
    {
        "id": "HA_REVIEW_001",
        "field": "sudden_peak_within_5_minutes",
        "reason": "Headache reached peak intensity suddenly.",
        "source_id": "NICE_CG150_1_1_1",
    },
    {
        "id": "HA_REVIEW_002",
        "field": "new_neurological_change",
        "reason": "A new neurological change was reported.",
        "source_id": "NICE_CG150_1_1_1",
    },
    {
        "id": "HA_REVIEW_003",
        "field": "impaired_consciousness",
        "reason": "Impaired consciousness was reported.",
        "source_id": "NICE_CG150_1_1_1",
    },
    {
        "id": "HA_REVIEW_004",
        "field": "worsening_with_fever",
        "reason": "Headache is worsening with fever.",
        "source_id": "NICE_CG150_1_1_1",
    },
]


def evaluate_headache_review(checkin):
    headache = checkin.get("headache", {})

    if not isinstance(headache, dict):
        raise TypeError("'headache' must be a dictionary.")

    fields = ["present"] + [
        rule["field"] for rule in HEADACHE_REVIEW_RULES
    ]

    # "False" ဆိုတဲ့ string နဲ့ False boolean မတူပါ
    for field in fields:
        value = headache.get(field)
        if value is not None and type(value) is not bool:
            raise TypeError(
                f"headache.{field} must be True, False, or None."
            )

    present = headache.get("present")
    triggers = []
    unknowns = []
    inconsistencies = []

    for rule in HEADACHE_REVIEW_RULES:
        value = headache.get(rule["field"])

        if value is True:
            triggers.append({
                "rule_id": rule["id"],
                "field": rule["field"],
                "reason": rule["reason"],
                "source_id": rule["source_id"],
            })
        elif value is None:
            unknowns.append(f"headache.{rule['field']}")

    if present is None:
        unknowns.insert(0, "headache.present")

    # Contradictory input ကြောင့် positive trigger မပျောက်စေရပါ
    if present is False and triggers:
        inconsistencies.append(
            "Headache marked absent, but a headache-review "
            "feature was reported. Clarify the record."
        )

    # Positive trigger ကို missing information ထက် ဦးစားပေးပါ
    if triggers:
        status = "review_required"
        message = (
            "Prompt clinical assessment is required. "
            "Use the applicable mission medical escalation protocol."
        )
    elif present is None or (present is True and unknowns):
        status = "insufficient_information"
        message = (
            "Selected headache-review questions are incomplete. "
            "Do not interpret missing answers as negative."
        )
    else:
        status = "no_selected_trigger_identified"
        message = (
            "No trigger identified by this limited check. "
            "This does not exclude serious illness."
        )

    return {
        "ruleset_version": RULESET_METADATA["version"],
        "ruleset_status": RULESET_METADATA["status"],
        "status": status,
        "message": message,
        "triggered_rules": triggers,
        "unknown_fields": unknowns,
        "input_inconsistencies": inconsistencies,
        "screening_complete": False,
        "diagnosis": None,
        "disease_probability": None,
    }


# လက်ရှိ blank check-in ကို စမ်းပါ
result = evaluate_headache_review(checkin)
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "ruleset_version": "0.1-draft",
  "ruleset_status": "draft_not_clinically_reviewed",
  "status": "insufficient_information",
  "message": "Selected headache-review questions are incomplete. Do not interpret missing answers as negative.",
  "triggered_rules": [],
  "unknown_fields": [
    "headache.present",
    "headache.sudden_peak_within_5_minutes",
    "headache.new_neurological_change",
    "headache.impaired_consciousness",
    "headache.worsening_with_fever"
  ],
  "input_inconsistencies": [],
  "screening_complete": false,
  "diagnosis": null,
  "disease_probability": null
}


In [3]:
# Case 1: အချက်အလက် မရှိသေး
case_unknown = deepcopy(CHECKIN_TEMPLATE)

assert (
    evaluate_headache_review(case_unknown)["status"]
    == "insufficient_information"
)

# Case 2: Warning feature ရှိ၊ အခြားအချက်တွေ မသိသေး
case_review = deepcopy(CHECKIN_TEMPLATE)
case_review["headache"]["present"] = True
case_review["headache"]["sudden_peak_within_5_minutes"] = True

assert (
    evaluate_headache_review(case_review)["status"]
    == "review_required"
)

# Case 3: ရွေးထားသော warning features အားလုံး absent
case_negative = deepcopy(CHECKIN_TEMPLATE)
case_negative["headache"]["present"] = True

for rule in HEADACHE_REVIEW_RULES:
    case_negative["headache"][rule["field"]] = False

assert (
    evaluate_headache_review(case_negative)["status"]
    == "no_selected_trigger_identified"
)

# Case 4: Contradictory input က positive trigger ကို မဖျောက်ရ
case_conflict = deepcopy(case_review)
case_conflict["headache"]["present"] = False

conflict_result = evaluate_headache_review(case_conflict)

assert conflict_result["status"] == "review_required"
assert conflict_result["input_inconsistencies"]

print("All 4 software logic checks passed.")

All 4 software logic checks passed.


In [4]:
import math

# Environment linkage ကို အတည်ပြုရန် သီးခြား fields
# Unknown ကို False အဖြစ် မယူပါနဲ့
CHECKIN_TEMPLATE["environment_linkage"] = {
    "crew_exposure_confirmed": None,
    "location_match_confirmed": None,
    "symptom_exposure_time_overlap_confirmed": None,
    "measurement_quality_accepted": None,
}

checkin = deepcopy(CHECKIN_TEMPLATE)


def build_environment_evidence(checkin):
    headache = checkin.get("headache", {})
    env = checkin.get("environment", {})
    linkage = checkin.get("environment_linkage", {})

    supporting = []
    conflicting = []
    unknowns = []

    # Linkage fields ကို strict boolean / None ပဲ လက်ခံပါ
    for field in CHECKIN_TEMPLATE["environment_linkage"]:
        value = linkage.get(field)

        if value is not None and type(value) is not bool:
            raise TypeError(
                f"environment_linkage.{field} "
                "must be True, False, or None."
            )

        if value is True:
            supporting.append(field)
        elif value is False:
            conflicting.append(field)
        else:
            unknowns.append(f"environment_linkage.{field}")

    co2 = env.get("co2_ppm")

    co2_valid = (
        type(co2) in (int, float)
        and math.isfinite(co2)
        and co2 >= 0
    )

    if co2_valid:
        supporting.append(f"CO2 measurement available: {co2} ppm")
    elif co2 is None:
        unknowns.append("environment.co2_ppm")
    else:
        conflicting.append("Invalid CO2 measurement")

    # Provenance မရှိရင် crew exposure evidence မပြည့်စုံပါ
    for field in [
        "source_id",
        "measured_at_utc",
        "location_id",
        "exposure_duration_minutes",
    ]:
        if env.get(field) is None:
            unknowns.append(f"environment.{field}")

    simulated = env.get("is_simulated")

    if simulated is not None and type(simulated) is not bool:
        raise TypeError("environment.is_simulated must be bool or None.")

    if simulated is None:
        unknowns.append("environment.is_simulated")

    if headache.get("present") is not True:
        status = "headache_not_confirmed"
    elif simulated is True:
        status = "simulation_only"
    elif conflicting:
        status = "linkage_not_established"
    elif unknowns:
        status = "insufficient_information"
    else:
        status = "context_available_for_review"

    return {
        "rule_id": "HA_ENV_CONTEXT_001",
        "rule_status": "draft_not_clinically_reviewed",
        "status": status,
        "supporting_evidence": supporting,
        "conflicting_or_invalid_evidence": conflicting,
        "unknown_fields": unknowns,

        # ML flag ကို environmental context အဖြစ်သာ ထိန်းထားပါ
        "environment_anomaly_flag": env.get("anomaly_flag"),

        # Exposure threshold rule ကို မတည်ဆောက်ရသေးပါ
        "co2_level_interpretation": "not_assessed",
        "causal_link": "not_established",
        "diagnosis": None,
        "disease_probability": None,
        "source_id": "NASA_CO2_REV_D",
    }


environment_result = build_environment_evidence(checkin)

print(json.dumps(
    environment_result,
    indent=2,
    ensure_ascii=False,
))

{
  "rule_id": "HA_ENV_CONTEXT_001",
  "rule_status": "draft_not_clinically_reviewed",
  "status": "headache_not_confirmed",
  "supporting_evidence": [],
  "conflicting_or_invalid_evidence": [],
  "unknown_fields": [
    "environment_linkage.crew_exposure_confirmed",
    "environment_linkage.location_match_confirmed",
    "environment_linkage.symptom_exposure_time_overlap_confirmed",
    "environment_linkage.measurement_quality_accepted",
    "environment.co2_ppm",
    "environment.source_id",
    "environment.measured_at_utc",
    "environment.location_id",
    "environment.exposure_duration_minutes",
    "environment.is_simulated"
  ],
  "environment_anomaly_flag": null,
  "co2_level_interpretation": "not_assessed",
  "causal_link": "not_established",
  "diagnosis": null,
  "disease_probability": null,
  "source_id": "NASA_CO2_REV_D"
}


In [5]:
# စမ်းသပ်ဖို့သာ ဖန်တီးထားသော inputs ဖြစ်သည်။
# Real crew / real exposure records မဟုတ်ပါ။

# Case 1: Headache ရှိပေမယ့် environment information မပြည့်
case_incomplete = deepcopy(CHECKIN_TEMPLATE)
case_incomplete["headache"]["present"] = True

assert (
    build_environment_evidence(case_incomplete)["status"]
    == "insufficient_information"
)

# Case 2: Simulated environment ဖြစ်ကြောင်း ဖော်ပြထား
case_simulated = deepcopy(case_incomplete)
case_simulated["environment"]["is_simulated"] = True

assert (
    build_environment_evidence(case_simulated)["status"]
    == "simulation_only"
)

# Case 3: နေရာမကိုက်ညီကြောင်း အတည်ပြုထား
case_mismatch = deepcopy(case_incomplete)
case_mismatch["environment"]["is_simulated"] = False
case_mismatch["environment_linkage"][
    "location_match_confirmed"
] = False

assert (
    build_environment_evidence(case_mismatch)["status"]
    == "linkage_not_established"
)

# Case 4: Context ပြည့်စုံသည့် branch ကို စမ်းရန် fixture
# is_simulated=False က software branch စမ်းရန်သာဖြစ်သည်။
# ဒီ fixture ကို actual measurement အဖြစ် မသိမ်းပါနှင့်။
case_linked = deepcopy(CHECKIN_TEMPLATE)
case_linked["headache"]["present"] = True

case_linked["environment"].update({
    "measured_at_utc": "2026-01-01T10:00:00Z",
    "location_id": "TEST_MODULE",
    "source_id": "UNIT_TEST_ONLY",
    "is_simulated": False,
    "co2_ppm": 4000,
    "exposure_duration_minutes": 30,
})

for field in case_linked["environment_linkage"]:
    case_linked["environment_linkage"][field] = True

linked_result = build_environment_evidence(case_linked)

assert linked_result["status"] == "context_available_for_review"

# Context ပြည့်စုံရုံဖြင့် cause / disease မထုတ်ရပါ
assert linked_result["causal_link"] == "not_established"
assert linked_result["co2_level_interpretation"] == "not_assessed"
assert linked_result["disease_probability"] is None

# Case 5: ML anomaly flag တစ်ခုတည်းက cause မဖြစ်ရပါ
case_anomaly_only = deepcopy(case_incomplete)
case_anomaly_only["environment"]["anomaly_flag"] = True

anomaly_result = build_environment_evidence(case_anomaly_only)

assert anomaly_result["status"] == "insufficient_information"
assert anomaly_result["causal_link"] == "not_established"

# Case 6: Negative CO2 ကို လက်မခံရပါ
case_invalid = deepcopy(case_linked)
case_invalid["environment"]["co2_ppm"] = -100

assert (
    build_environment_evidence(case_invalid)["status"]
    == "linkage_not_established"
)

print("All 6 environment logic checks passed.")

All 6 environment logic checks passed.


In [6]:
SOURCES["NICE_CG150_TABLE1"] = {
    "title": "Headache features by headache type",
    "section": "Recommendation 1.2.1, Table 1",
    "url": (
        "https://www.nice.org.uk/guidance/"
        "cg150/chapter/Recommendations"
    ),
    "scope": "Partial phenotype reference; not spaceflight-validated",
}

# ရှိပြီးသား input values တွေကို မဖျက်ပါ
NEW_HEADACHE_FIELDS = {
    "pain_location": None,       # unilateral / bilateral / other
    "pain_quality": None,        # pulsating / pressing_tightening / other
    "pain_intensity": None,      # mild / moderate / severe / very_severe
    "routine_activity_worsens_or_is_avoided": None,
    "nausea": None,
    "vomiting": None,
    "light_sensitivity": None,
    "sound_sensitivity": None,
}

for field, default in NEW_HEADACHE_FIELDS.items():
    CHECKIN_TEMPLATE["headache"].setdefault(field, default)
    checkin["headache"].setdefault(field, default)

PATTERNS = [
    {
        "id": "HA_PATTERN_MIGRAINE_001",
        "label": "Migraine-like pattern",
        "features": {
            "pain_location": {"unilateral", "bilateral"},
            "pain_quality": {"pulsating"},
            "pain_intensity": {"moderate", "severe"},
            "routine_activity_worsens_or_is_avoided": {True},
        },
        "associated_symptom_group": "any_present",
    },
    {
        "id": "HA_PATTERN_TENSION_001",
        "label": "Tension-type-like pattern",
        "features": {
            "pain_location": {"bilateral"},
            "pain_quality": {"pressing_tightening"},
            "pain_intensity": {"mild", "moderate"},
            "routine_activity_worsens_or_is_avoided": {False},
        },
        "associated_symptom_group": "all_absent",
    },
]

print("Pattern definitions ready.")

Pattern definitions ready.


In [10]:
def build_headache_pattern_cards(checkin):
    review = evaluate_headache_review(checkin)
    h = checkin.get("headache", {})

    result = {
        "status": None,
        "review_check": review,
        "pattern_cards": [],
        "ranking": None,
        "disease_probabilities": None,
        "diagnosis": None,
        "other_conditions_not_excluded": True,
        "clinical_review_required": True,
    }

    # Positive warning / unanswered review questions ရှိရင်
    # Pattern cards ထုတ်ခြင်းကို ခဏရပ်ပါ
    if review["status"] != "no_selected_trigger_identified":
        result["status"] = "pattern_review_deferred"
        return result

    if h.get("present") is not True:
        result["status"] = "headache_not_present"
        return result

    enum_fields = {
        "pain_location": {"unilateral", "bilateral", "other"},
        "pain_quality": {"pulsating", "pressing_tightening", "other"},
        "pain_intensity": {
            "mild", "moderate", "severe", "very_severe"
        },
    }

    symptom_fields = [
        "nausea",
        "vomiting",
        "light_sensitivity",
        "sound_sensitivity",
    ]

    # Invalid values ကို negative evidence အဖြစ် မယူပါ
    for field, allowed in enum_fields.items():
        value = h.get(field)
        if value is not None and value not in allowed:
            raise ValueError(f"Invalid headache.{field}: {value!r}")

    boolean_fields = symptom_fields + [
        "routine_activity_worsens_or_is_avoided"
    ]

    for field in boolean_fields:
        value = h.get(field)
        if value is not None and type(value) is not bool:
            raise TypeError(
                f"headache.{field} must be True, False, or None."
            )

    for pattern in PATTERNS:
        supporting = []
        conflicting = []
        missing = []

        for field, expected in pattern["features"].items():
            value = h.get(field)

            if value is None:
                missing.append(f"headache.{field}")
            elif value in expected:
                supporting.append({
                    "field": field,
                    "observed": value,
                })
            else:
                conflicting.append({
                    "field": field,
                    "observed": value,
                    "meaning": "Differs from this reference pattern",
                })

        present_symptoms = [
            field for field in symptom_fields
            if h.get(field) is True
        ]
        unknown_symptoms = [
            field for field in symptom_fields
            if h.get(field) is None
        ]

        if pattern["associated_symptom_group"] == "any_present":
            if present_symptoms:
                supporting.append({
                    "field": "associated_symptom_group",
                    "observed": present_symptoms,
                })
            elif unknown_symptoms:
                missing.extend(
                    f"headache.{field}"
                    for field in unknown_symptoms
                )
            else:
                conflicting.append({
                    "field": "associated_symptom_group",
                    "observed": "Selected symptoms all reported absent",
                })

        else:
            if present_symptoms:
                conflicting.append({
                    "field": "associated_symptom_group",
                    "observed": present_symptoms,
                })

            if unknown_symptoms:
                missing.extend(
                    f"headache.{field}"
                    for field in unknown_symptoms
                )
            elif not present_symptoms:
                supporting.append({
                    "field": "associated_symptom_group",
                    "observed": "Selected symptoms all reported absent",
                })
                # Group ကို အခြား symptom နဲ့ support လုပ်နိုင်သော်လည်း
        # unanswered individual questions ကို မဖျောက်ပါနှင့်
        missing = sorted(set(
            missing
            + [
                f"headache.{field}"
                for field in symptom_fields
                if h.get(field) is None
            ]
        ))

        result["pattern_cards"].append({
            "rule_id": pattern["id"],
            "label": pattern["label"],
            "status": "partial_evidence_only",
            "supporting_evidence": supporting,
            "conflicting_evidence": conflicting,
            "missing_fields": missing,
            "not_assessed": [
                "Full episode-duration criteria",
                "Attack frequency and recurrence",
                "Medication effects and treatment history",
                "Complete secondary-headache assessment",
                "Other headache types and spaceflight causes",
            ],
            "source_id": "NICE_CG150_TABLE1",
            "clinically_reviewed": False,
        })

    result["status"] = "partial_pattern_evidence_available"
    return result

In [8]:
demo = deepcopy(CHECKIN_TEMPLATE)
demo["crew_id"] = "DEMO_ONLY"

demo["headache"].update({
    "present": True,
    "pain_location": "unilateral",
    "pain_quality": "pulsating",
    "pain_intensity": "moderate",
    "routine_activity_worsens_or_is_avoided": True,
    "nausea": True,
    "vomiting": False,
    "light_sensitivity": True,
    "sound_sensitivity": None,
})

# Unit-test example အတွက်သာ negative answers သတ်မှတ်ထားသည်။
# Actual user answers မရှိဘဲ ဒီလို False မဖြည့်ရပါ။
for rule in HEADACHE_REVIEW_RULES:
    demo["headache"][rule["field"]] = False

pattern_result = build_headache_pattern_cards(demo)

print(json.dumps(
    pattern_result,
    indent=2,
    ensure_ascii=False,
))

{
  "status": "partial_pattern_evidence_available",
  "review_check": {
    "ruleset_version": "0.1-draft",
    "ruleset_status": "draft_not_clinically_reviewed",
    "status": "no_selected_trigger_identified",
    "message": "No trigger identified by this limited check. This does not exclude serious illness.",
    "triggered_rules": [],
    "unknown_fields": [],
    "input_inconsistencies": [],
    "screening_complete": false,
    "diagnosis": null,
    "disease_probability": null
  },
  "pattern_cards": [
    {
      "rule_id": "HA_PATTERN_MIGRAINE_001",
      "label": "Migraine-like pattern",
      "status": "partial_evidence_only",
      "supporting_evidence": [
        {
          "field": "pain_location",
          "observed": "unilateral"
        },
        {
          "field": "pain_quality",
          "observed": "pulsating"
        },
        {
          "field": "pain_intensity",
          "observed": "moderate"
        },
        {
          "field": "routine_activity_worse

In [11]:
def get_card(result, rule_id):
    return next(
        card for card in result["pattern_cards"]
        if card["rule_id"] == rule_id
    )


# 1. Unknown answer ကို card နှစ်ခုလုံးမှာ ထိန်းထားရပါမယ်
r = build_headache_pattern_cards(demo)

for card in r["pattern_cards"]:
    assert "headache.sound_sensitivity" in card["missing_fields"]

assert r["diagnosis"] is None
assert r["disease_probabilities"] is None
assert r["ranking"] is None


# 2. Warning feature ရှိရင် pattern cards မထုတ်ရပါ
warning_case = deepcopy(demo)
warning_case["headache"]["new_neurological_change"] = True

r = build_headache_pattern_cards(warning_case)

assert r["status"] == "pattern_review_deferred"
assert r["review_check"]["status"] == "review_required"
assert r["pattern_cards"] == []


# 3. Review question တစ်ခု မဖြေထားလျှင် defer လုပ်ရပါ
incomplete_case = deepcopy(demo)
incomplete_case["headache"]["impaired_consciousness"] = None

r = build_headache_pattern_cards(incomplete_case)

assert r["status"] == "pattern_review_deferred"
assert r["review_check"]["status"] == "insufficient_information"


# 4. Tension-type-like features ပါသော fabricated example
tension_case = deepcopy(demo)
tension_case["headache"].update({
    "pain_location": "bilateral",
    "pain_quality": "pressing_tightening",
    "pain_intensity": "mild",
    "routine_activity_worsens_or_is_avoided": False,
    "nausea": False,
    "vomiting": False,
    "light_sensitivity": False,
    "sound_sensitivity": False,
})

r = build_headache_pattern_cards(tension_case)
card = get_card(r, "HA_PATTERN_TENSION_001")

assert card["supporting_evidence"]
assert card["conflicting_evidence"] == []
assert card["missing_fields"] == []
assert card["not_assessed"]  # Full assessment မပြီးသေးပါ


# 5. Boolean string ကို မလက်ခံရပါ
invalid_case = deepcopy(demo)
invalid_case["headache"]["nausea"] = "False"

try:
    build_headache_pattern_cards(invalid_case)
except TypeError:
    pass
else:
    raise AssertionError("Invalid boolean string was accepted.")


print("All 5 pattern logic checks passed.")

All 5 pattern logic checks passed.


In [12]:
# Existing answers တွေကို မဖျက်ပါ
for record in [CHECKIN_TEMPLATE, checkin]:
    record["headache"].setdefault(
        "episode_status", None
    )  # ongoing / resolved

    record["headache"].setdefault(
        "duration_interpretable_as_untreated", None
    )  # True / False / None

    record["history"].setdefault("age_years", None)
    record["history"].setdefault(
        "previous_headache_details", None
    )

print("Episode and history fields ready.")

Episode and history fields ready.


In [13]:
def add_episode_and_history_context(checkin):
    result = build_headache_pattern_cards(checkin)

    # Review gate ကြောင့် deferred ဖြစ်နေရင် ဆက်မတွက်ပါ
    if result["status"] != "partial_pattern_evidence_available":
        return result

    h = checkin["headache"]
    history = checkin.get("history", {})

    duration = h.get("duration_minutes")
    episode_status = h.get("episode_status")
    interpretable = h.get("duration_interpretable_as_untreated")
    age = history.get("age_years")
    previous_similar = history.get("previous_similar_headaches")

    # Input validation
    for name, value in [
        ("duration_minutes", duration),
        ("age_years", age),
    ]:
        if value is not None:
            if (
                type(value) not in (int, float)
                or not math.isfinite(value)
                or value < 0
            ):
                raise ValueError(f"{name} must be a finite nonnegative number.")

    if episode_status not in (None, "ongoing", "resolved"):
        raise ValueError("episode_status must be ongoing, resolved, or None.")

    for name, value in [
        ("duration_interpretable_as_untreated", interpretable),
        ("previous_similar_headaches", previous_similar),
    ]:
        if value is not None and type(value) is not bool:
            raise TypeError(f"{name} must be True, False, or None.")

    # History က context ဖြစ်သည်။ Diagnosis အတည်ပြုချက် မဟုတ်ပါ။
    result["history_context"] = {
        "age_years": age,
        "previous_similar_headaches": previous_similar,
        "previous_headache_details":
            history.get("previous_headache_details"),
        "known_conditions": history.get("known_conditions"),
        "current_medications": history.get("current_medications"),
        "allergies": history.get("allergies"),
        "history_used_to_exclude_other_causes": False,
    }

    for card in result["pattern_cards"]:
        context = {
            "episode_status": episode_status,
            "reported_duration_minutes": duration,
            "status": "not_assessed",
            "explanation": None,
            "source_id": "NICE_CG150_TABLE1",
        }

        missing = []

        if age is None:
            missing.append("history.age_years")
        if duration is None:
            missing.append("headache.duration_minutes")
        if episode_status is None:
            missing.append("headache.episode_status")
        if interpretable is None:
            missing.append(
                "headache.duration_interpretable_as_untreated"
            )

        if missing:
            context["status"] = "insufficient_information"

        elif age < 18:
            context["status"] = "outside_adult_duration_module"
            context["explanation"] = (
                "This draft duration comparison covers adults only."
            )

        elif interpretable is False:
            context["status"] = "duration_not_comparable"
            context["explanation"] = (
                "Do not compare this duration directly with "
                "untreated reference episode duration."
            )

        else:
            is_migraine = (
                card["rule_id"] == "HA_PATTERN_MIGRAINE_001"
            )

            minimum = 240 if is_migraine else 30
            maximum = 4320 if is_migraine else None

            if duration < minimum:
                if episode_status == "ongoing":
                    context["status"] = "episode_still_evolving"
                    context["explanation"] = (
                        "Minimum reference duration not yet reached. "
                        "Ongoing duration does not exclude this pattern."
                    )
                else:
                    context["status"] = "differs_from_reference_duration"
                    context["explanation"] = (
                        "Completed episode was shorter than the "
                        "reference range; this is not an exclusion rule."
                    )

            elif maximum is not None and duration > maximum:
                context["status"] = "differs_from_reference_duration"
                context["explanation"] = (
                    "Duration exceeds the reference range. "
                    "Clinical reassessment is needed."
                )

            else:
                context["status"] = "compatible_with_reference_duration"
                context["explanation"] = (
                    "Duration is compatible with this reference pattern. "
                    "It does not establish the diagnosis."
                )

        card["episode_context"] = context
        card["missing_fields"] = sorted(set(
            card["missing_fields"] + missing
        ))

        # Partial duration comparison ပါလာပြီဖြစ်ပေမယ့်
        # full clinical duration assessment ပြီးပြီဟု မဆိုပါ
        card["not_assessed"] = [
            item for item in card["not_assessed"]
            if item != "Full episode-duration criteria"
        ]
        card["not_assessed"].append(
            "Independent verification of episode timing "
            "and treatment effects"
        )

    return result

In [14]:
episode_demo = deepcopy(demo)

episode_demo["history"].update({
    "age_years": 28,
    "previous_similar_headaches": True,
    "previous_headache_details":
        "Fabricated example: similar episodes reported previously.",
})

episode_demo["headache"].update({
    "duration_minutes": 60,
    "episode_status": "ongoing",
    "duration_interpretable_as_untreated": True,
})

episode_result = add_episode_and_history_context(episode_demo)

for card in episode_result["pattern_cards"]:
    print(card["label"])
    print(json.dumps(
        card["episode_context"],
        indent=2,
        ensure_ascii=False,
    ))
    print()

Migraine-like pattern
{
  "episode_status": "ongoing",
  "reported_duration_minutes": 60,
  "status": "episode_still_evolving",
  "explanation": "Minimum reference duration not yet reached. Ongoing duration does not exclude this pattern.",
  "source_id": "NICE_CG150_TABLE1"
}

Tension-type-like pattern
{
  "episode_status": "ongoing",
  "reported_duration_minutes": 60,
  "status": "compatible_with_reference_duration",
  "explanation": "Duration is compatible with this reference pattern. It does not establish the diagnosis.",
  "source_id": "NICE_CG150_TABLE1"
}



In [15]:
def migraine_duration_status(record):
    result = add_episode_and_history_context(record)
    card = get_card(result, "HA_PATTERN_MIGRAINE_001")
    return card["episode_context"]["status"]


# duration, episode status, expected result
duration_cases = [
    (60, "ongoing", "episode_still_evolving"),
    (60, "resolved", "differs_from_reference_duration"),
    (240, "resolved", "compatible_with_reference_duration"),
    (4320, "resolved", "compatible_with_reference_duration"),
    (4321, "ongoing", "differs_from_reference_duration"),
]

for duration, episode_status, expected in duration_cases:
    case = deepcopy(episode_demo)

    case["headache"].update({
        "duration_minutes": duration,
        "episode_status": episode_status,
        "duration_interpretable_as_untreated": True,
    })

    actual = migraine_duration_status(case)

    assert actual == expected, (
        f"{duration} min / {episode_status}: "
        f"expected {expected}, got {actual}"
    )


# Treatment effects ကြောင့် duration မနှိုင်းသင့်သည့် case
case_treated = deepcopy(episode_demo)
case_treated["headache"][
    "duration_interpretable_as_untreated"
] = False

assert (
    migraine_duration_status(case_treated)
    == "duration_not_comparable"
)


# Age မသိသေးလျှင် adult criteria မသုံးရပါ
case_unknown_age = deepcopy(episode_demo)
case_unknown_age["history"]["age_years"] = None

assert (
    migraine_duration_status(case_unknown_age)
    == "insufficient_information"
)


# Adult module ပြင်ပ
case_under_18 = deepcopy(episode_demo)
case_under_18["history"]["age_years"] = 17

assert (
    migraine_duration_status(case_under_18)
    == "outside_adult_duration_module"
)


# Invalid duration ကို မလက်ခံရပါ
case_invalid = deepcopy(episode_demo)
case_invalid["headache"]["duration_minutes"] = -10

try:
    add_episode_and_history_context(case_invalid)
except ValueError:
    pass
else:
    raise AssertionError("Negative duration was accepted.")


print("All 9 episode-context logic checks passed.")

All 9 episode-context logic checks passed.


In [16]:
from datetime import datetime, timezone
from uuid import uuid4


def build_health_review_report(checkin):
    # Existing functions ကို ခေါ်သုံးခြင်းသာဖြစ်သည်
    # Clinical reasoning အသစ် မထည့်ထားပါ
    pattern_result = add_episode_and_history_context(checkin)
    environment_result = build_environment_evidence(checkin)

    review = pattern_result["review_check"]

    if review["status"] == "review_required":
        workflow_status = "medical_review_required"
    elif review["status"] == "insufficient_information":
        workflow_status = "additional_review_answers_required"
    elif checkin.get("headache", {}).get("present") is not True:
        workflow_status = "no_headache_pattern_report"
    else:
        workflow_status = "draft_evidence_ready_for_review"

    unknown_fields = set(review["unknown_fields"])
    unknown_fields.update(environment_result["unknown_fields"])

    for card in pattern_result["pattern_cards"]:
        unknown_fields.update(card["missing_fields"])

    # Metadata / history မပြည့်မှုကိုလည်း ပြပါ
    for field in ["crew_id", "recorded_at_utc"]:
        if checkin.get(field) is None:
            unknown_fields.add(field)

    history = checkin.get("history", {})

    for field in [
        "age_years",
        "previous_similar_headaches",
        "known_conditions",
        "current_medications",
        "allergies",
    ]:
        if history.get(field) is None:
            unknown_fields.add(f"history.{field}")

    return {
        "report_id": str(uuid4()),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "report_type": "headache_evidence_review",
        "schema_version": "0.1",

        "crew_id": checkin.get("crew_id"),
        "checkin_recorded_at_utc": checkin.get("recorded_at_utc"),

        "ruleset": deepcopy(RULESET_METADATA),
        "workflow_status": workflow_status,

        "review_checks": review,
        "pattern_review_status": pattern_result["status"],
        "pattern_cards": pattern_result["pattern_cards"],

        "history_context": {
            "reported_history": deepcopy(history),
            "independently_verified": False,
        },

        # Environment context သည် diagnostic card မဟုတ်ပါ
        "environment_evidence": environment_result,

        "unknown_fields": sorted(unknown_fields),
        "source_registry": deepcopy(SOURCES),

        "diagnosis": None,
        "disease_probabilities": None,
        "condition_ranking": None,
        "medication_recommendations": None,

        "limitations": [
            "Draft rules have not been clinically reviewed.",
            "Selected review checks are not a complete screening protocol.",
            "Pattern cards are partial evidence summaries.",
            "Environmental context does not establish causation.",
            "This JSON schema is not a verified NASA exchange format.",
        ],
    }


# Fabricated example ကိုသာ စမ်းပါ
report_demo = deepcopy(episode_demo)

report_demo["recorded_at_utc"] = "2026-01-01T10:00:00Z"
report_demo["environment"]["is_simulated"] = True

health_report = build_health_review_report(report_demo)

print(json.dumps(
    health_report,
    indent=2,
    ensure_ascii=False,
))

{
  "report_id": "41856220-88a0-41fe-a9f9-9330f4ea5027",
  "created_at_utc": "2026-11-08T06:30:49.331698+00:00",
  "report_type": "headache_evidence_review",
  "schema_version": "0.1",
  "crew_id": "DEMO_ONLY",
  "checkin_recorded_at_utc": "2026-01-01T10:00:00Z",
  "ruleset": {
    "name": "HabitatPulse Headache Review",
    "version": "0.1-draft",
    "status": "draft_not_clinically_reviewed",
    "purpose": "Evidence organization and medical review support",
    "validated_for_spaceflight": false,
    "produces_disease_probabilities": false
  },
  "workflow_status": "draft_evidence_ready_for_review",
  "review_checks": {
    "ruleset_version": "0.1-draft",
    "ruleset_status": "draft_not_clinically_reviewed",
    "status": "no_selected_trigger_identified",
    "message": "No trigger identified by this limited check. This does not exclude serious illness.",
    "triggered_rules": [],
    "unknown_fields": [],
    "input_inconsistencies": [],
    "screening_complete": false,
    "diag

In [17]:
json_text = json.dumps(
    health_report,
    ensure_ascii=False,
    indent=2,
    allow_nan=False,
)

restored_report = json.loads(json_text)

assert restored_report == health_report
assert restored_report["diagnosis"] is None
assert restored_report["condition_ranking"] is None

# Warning feature ရှိရင် report ထဲမှာလည်း defer ဖြစ်ရပါမယ်
warning_demo = deepcopy(report_demo)
warning_demo["headache"]["new_neurological_change"] = True

warning_report = build_health_review_report(warning_demo)

assert warning_report["workflow_status"] == "medical_review_required"
assert warning_report["pattern_cards"] == []

print("Report serialization and review-gate checks passed.")

Report serialization and review-gate checks passed.


In [18]:
from pathlib import Path
import hashlib
import json

# 1. Source URLs ကို plain strings အဖြစ် ပြင်ပါ
nice_url = (
    "https://www.nice.org.uk/guidance/"
    "cg150/chapter/Recommendations"
)

nasa_url = (
    "https://www.nasa.gov/wp-content/"
    "uploads/2023/12/ochmo-tb-004-carbon-dioxide.pdf"
)

SOURCES["NICE_CG150_1_1_1"]["url"] = nice_url
SOURCES["NICE_CG150_TABLE1"]["url"] = nice_url
SOURCES["NASA_CO2_REV_D"]["url"] = nasa_url

# Updated sources ပါအောင် report ပြန်ထုတ်ပါ
health_report = build_health_review_report(report_demo)

# 2. Report ပြန်စမ်းနိုင်ဖို့ မူရင်း input ကိုပါ သိမ်းပါ
checkpoint = {
    "checkpoint_version": "0.1",
    "purpose": "Synthetic development example",
    "is_fabricated_test_case": True,
    "input_snapshot": deepcopy(report_demo),
    "output_report": health_report,
}

checkpoint_dir = Path("healthcare_rule_outputs")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = (
    checkpoint_dir
    / f"headache_demo_{health_report['report_id']}.json"
)

# Existing file ကို မတော်တဆ overwrite မလုပ်ပါ
with checkpoint_path.open("x", encoding="utf-8") as f:
    json.dump(
        checkpoint,
        f,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

# 3. သိမ်းထားသောဖိုင်ကို ပြန်ဖတ်ပြီး တူညီကြောင်း စစ်ပါ
with checkpoint_path.open(encoding="utf-8") as f:
    restored = json.load(f)

assert restored == checkpoint

# 4. File integrity စစ်ရန် checksum သိမ်းပါ
digest = hashlib.sha256(
    checkpoint_path.read_bytes()
).hexdigest()

checksum_path = checkpoint_path.with_suffix(".json.sha256")

checksum_path.write_text(
    f"{digest}  {checkpoint_path.name}\n",
    encoding="utf-8",
)

print("Checkpoint saved:", checkpoint_path.resolve())
print("Read-back verification: PASSED")
print("SHA-256:", digest)

Checkpoint saved: E:\Nasa space App\healthcare_rule_outputs\headache_demo_35a0a9c0-46f1-4d22-857c-e1a746fec3d4.json
Read-back verification: PASSED
SHA-256: 83e6800828dbaccee60786753160a9b04af38c6d2477967631a7f76cdbd90cab
